# 2 — v2 Extraction Pipeline

Runs the **v2 analyzer** on all 5 PDFs via the enhanced classifier,
saves JSON + Excel outputs.

v2 schema fields per row:
- `lineItem` (extract) — row label verbatim
- `level` (classify, 0-3) — indentation depth
- `isSectionHeader` (classify, true/false)
- `isSubtotal` (classify, true/false)
- `values` (extract array) — cell values left-to-right

In [ ]:
import json, os, re, time
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")

endpoint = os.environ["FOUNDRY_ENDPOINT"].split("/api/projects/")[0].rstrip("/") + "/"
credential = ClientSecretCredential(
    tenant_id=os.environ["AZURE_TENANT_ID"],
    client_id=os.environ["AZURE_CLIENT_ID"],
    client_secret=os.environ["AZURE_CLIENT_SECRET"],
)
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version="2025-11-01")
#print("Client ready:", endpoint)

## Deploy v2 Analyzer & Classifier

In [ ]:
# Deploy the v2 financial-table extraction analyzer
ANALYZER_V2_ID = "sec_financial_tables_v2"
v2_path = REPO_ROOT / "analyzers" / f"{ANALYZER_V2_ID}.json"
v2_tmpl = json.loads(v2_path.read_text(encoding="utf-8"))
v2_tmpl["models"] = {
    "completion": os.environ["GPT41_MODEL_DEPLOYMENT"],
    "embedding": os.environ["EMBEDDING_MODEL_DEPLOYMENT"],
}

t0 = time.time()
client.begin_create_analyzer(analyzer_id=ANALYZER_V2_ID, resource=v2_tmpl, allow_replace=True).result()
print(f"Analyzer '{ANALYZER_V2_ID}' deployed in {time.time() - t0:.1f}s")

In [ ]:
# Deploy the v2 enhanced classifier (same structure, but pointing to v2 analyzer)
CLASSIFIER_V2_ID = "sec_fs_classifier_v2"

classifier_v2_template = {
    "baseAnalyzerId": "prebuilt-document",
    "description": (
        "SEC 10-K/10-Q financial statement classifier (v2). "
        "Segments filings into the five primary consolidated financial statements "
        "and routes each to the v2 extraction analyzer (reduced schema)."
    ),
    "config": {
        "returnDetails": True,
        "enableSegment": True,
        "contentCategories": {
            "BalanceSheet": {
                "description": (
                    "Consolidated Balance Sheets or Statement of Financial Position. "
                    "Shows total assets, total liabilities, and stockholders' equity "
                    "at specific reporting dates. Typically titled "
                    "'Consolidated Balance Sheets'."
                ),
                "analyzerId": ANALYZER_V2_ID,
            },
            "IncomeStatement": {
                "description": (
                    "Consolidated Statements of Operations, Statements of Income, "
                    "or Income Statements. Shows revenues, costs, expenses, and "
                    "net income/loss over reporting periods. "
                    "Does NOT include Statements of Comprehensive Income."
                ),
                "analyzerId": ANALYZER_V2_ID,
            },
            "ComprehensiveIncome": {
                "description": (
                    "Consolidated Statements of Comprehensive Income (Loss). "
                    "Shows net income plus other comprehensive income items such as "
                    "unrealized gains/losses, foreign currency translation, "
                    "and pension adjustments."
                ),
                "analyzerId": ANALYZER_V2_ID,
            },
            "Equity": {
                "description": (
                    "Consolidated Statements of Changes in Stockholders' Equity. "
                    "Shows changes in equity components (common stock, APIC, "
                    "retained earnings, AOCI, treasury stock) across periods. "
                    "May contain multiple sub-tables for different comparison periods."
                ),
                "analyzerId": ANALYZER_V2_ID,
            },
            "CashFlow": {
                "description": (
                    "Consolidated Statements of Cash Flows. "
                    "Shows operating, investing, and financing activities "
                    "with beginning and ending cash balances."
                ),
                "analyzerId": ANALYZER_V2_ID,
            },
            "Other": {
                "description": (
                    "Any page that is NOT one of the five primary financial "
                    "statement tables. Includes: cover pages, table of contents, "
                    "MD&A, notes to financial statements, auditor reports, "
                    "exhibits, and all other content."
                ),
            },
        },
    },
    "models": {
        "completion": os.environ["GPT41_MODEL_DEPLOYMENT"],
        "embedding": os.environ["EMBEDDING_MODEL_DEPLOYMENT"],
    },
    "tags": {"purpose": "sec-fs-classifier", "version": "v2"},
}

t0 = time.time()
client.begin_create_analyzer(
    analyzer_id=CLASSIFIER_V2_ID, resource=classifier_v2_template, allow_replace=True
).result()
print(f"Classifier '{CLASSIFIER_V2_ID}' deployed in {time.time() - t0:.1f}s")

## Classify & Extract (v2)

Process all 5 PDFs through the v2 classifier in parallel.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

pdf_dir = REPO_ROOT / "email" / "attachements"
pdfs = sorted(pdf_dir.glob("*.pdf"))
print(f"Found {len(pdfs)} PDFs — processing in parallel with v2 classifier\n")

MAX_RETRIES = 2  # retry up to 2 extra times if 0 rows detected

def _has_empty_tables(result):
    """Return list of segment categories whose rows.valueArray is None/empty."""
    empties = []
    for seg in result.get("contents", []):
        cat = seg.get("category", "Other")
        if cat == "Other":
            continue
        ft = seg.get("fields", {}).get("financialTables", {})
        for tbl in ft.get("valueArray", []):
            tobj = tbl.get("valueObject", {})
            rows = (tobj.get("rows") or {}).get("valueArray") or []
            if len(rows) == 0:
                empties.append(cat)
    return empties

def process_pdf_v2(pdf_path):
    pdf_bytes = pdf_path.read_bytes()
    attempt = 0
    while True:
        t0 = time.time()
        poller = client.begin_analyze_binary(
            analyzer_id=CLASSIFIER_V2_ID,
            binary_input=pdf_bytes,
            content_type="application/pdf",
        )
        result = poller.result().as_dict()
        elapsed = time.time() - t0
        empties = _has_empty_tables(result)
        attempt += 1
        if not empties or attempt > MAX_RETRIES:
            break
        print(f"  ⚠ {pdf_path.stem}: 0-row tables in {empties}, retry {attempt}/{MAX_RETRIES}...")

    contents = result.get("contents", [])
    cats = {}
    for seg in contents:
        cat = seg.get("category", "Unknown")
        cats[cat] = cats.get(cat, 0) + 1
    retry_note = f" (after {attempt} attempts)" if attempt > 1 else ""
    return pdf_path.stem, result, elapsed, cats, retry_note

v2_results = {}
t_total = time.time()

with ThreadPoolExecutor(max_workers=len(pdfs)) as executor:
    futures = {executor.submit(process_pdf_v2, pdf): pdf for pdf in pdfs}
    for future in as_completed(futures):
        stem, result, elapsed, cats, retry_note = future.result()
        v2_results[stem] = result
        cat_str = ", ".join(f"{k}:{v}" for k, v in sorted(cats.items()))
        n_seg = sum(cats.values())
        print(f"  {stem}: {n_seg} segments, {elapsed:.0f}s  [{cat_str}]{retry_note}")

print(f"\nDone. Total wall time: {time.time() - t_total:.0f}s")

In [ ]:
# Display v2 classification and extraction results
EXPECTED_FS = ["BalanceSheet", "IncomeStatement", "ComprehensiveIncome", "Equity", "CashFlow"]

for stem, result in sorted(v2_results.items()):
    contents = result.get("contents", [])
    print(f"\n{'='*80}")
    print(f"{stem}")
    print(f"{'='*80}")

    for seg in contents:
        cat = seg.get("category", "Unknown")
        if cat == "Other":
            continue
        pg_start = seg.get("startPageNumber", "?")
        pg_end = seg.get("endPageNumber", "?")
        fields = seg.get("fields", {})
        ft = fields.get("financialTables", {})
        tables = ft.get("valueArray", [])
        print(f"\n  {cat} (pages {pg_start}-{pg_end}): {len(tables)} table(s)")
        for j, tbl_raw in enumerate(tables, 1):
            tobj = tbl_raw.get("valueObject", {})
            title = (tobj.get("tableTitle") or {}).get("valueString", "")
            stype = (tobj.get("statementType") or {}).get("valueString", "")
            rows = (tobj.get("rows") or {}).get("valueArray", [])
            hdrs = (tobj.get("periodHeaders") or {}).get("valueArray", [])
            unit = (tobj.get("unit") or {}).get("valueString", "")
            # Count empty lineItems
            empty = sum(1 for r in rows if not (r.get("valueObject", {}).get("lineItem") or {}).get("valueString", "").strip())
            status = f" ⚠ {empty} empty lineItems" if empty else ""
            print(f"    T{j}: {stype} | {len(rows)} rows, {len(hdrs)} cols | {unit} | {title[:60]}{status}")

    found = {seg.get("category") for seg in contents if seg.get("category") != "Other"}
    missing = [fs for fs in EXPECTED_FS if fs not in found]
    if missing:
        print(f"\n  MISSING: {', '.join(missing)}")
    else:
        print(f"\n  All 5 FS types found")

## Save v2 Results

In [ ]:
import sys, importlib
sys.path.insert(0, str(REPO_ROOT / "scripts"))
if "export_to_excel" in sys.modules:
    importlib.reload(sys.modules["export_to_excel"])
from export_to_excel import export_document

out_dir = REPO_ROOT / "output"
v2_out_dir = out_dir / "v2"
v2_out_dir.mkdir(parents=True, exist_ok=True)
v2_excel_dir = v2_out_dir / "excel"

for stem, result in v2_results.items():
    # Save raw classifier result
    raw_path = v2_out_dir / f"{stem}_v2_classified_raw.json"
    raw_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")

    # Merge all FS segments' financialTables into one list for export
    merged_tables = []
    for seg in result.get("contents", []):
        if seg.get("category", "Other") == "Other":
            continue
        ft = seg.get("fields", {}).get("financialTables", {})
        merged_tables.extend(ft.get("valueArray", []))

    merged = {
        "contents": [{
            "fields": {
                "financialTables": {
                    "type": "array",
                    "valueArray": merged_tables,
                }
            }
        }]
    }
    json_path = v2_out_dir / f"{stem}_v2_classified.json"
    json_path.write_text(json.dumps(merged, indent=2, ensure_ascii=False), encoding="utf-8")

    # Export to Excel
    xlsx_path = export_document(json_path, v2_excel_dir)
    print(f"{stem}: {len(merged_tables)} tables -> {json_path.name}, {xlsx_path.name}")